# 01 — Datos + EDA

Spec: `docs/superpowers/specs/2026-10-17-datos-eda-design.md`. El notebook solo llama a funciones de
`src/credit_risk/`; no contiene lógica propia ni muestra filas de datos (solo agregados y gráficos).

- **Fase 1:** calidad de datos sobre `cs-training.csv` completo.
- **Split:** deduplicación exacta y partición estratificada train / valid / test.
- **Fase 2:** EDA estadístico **solo sobre el train interno**.

## 0. Setup

In [1]:
import pandas as pd
from IPython.display import display

from credit_risk.config import load_config
from credit_risk.data import quality
from credit_risk.data.load import load_raw_training
from credit_risk.data.schema import FEATURES, RATIO_COLUMNS
from credit_risk.data.split import (
    cross_partition_feature_overlap,
    partition_summary,
    save_partitions,
    split_train_valid_test,
)
from credit_risk.eda import plots, profile

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
cfg = load_config()
raw = load_raw_training(cfg.training_file)
FIG_DIR = cfg.paths.reports / "figures"
print(f"Filas: {len(raw):,} · columnas: {raw.shape[1]}")

Filas: 150,000 · columnas: 11


## 1. Fase 1 — Calidad de datos (archivo completo)

### Esquema, tipos y formato

In [2]:
print(quality.check_schema(raw))
display(quality.dtype_report(raw))
display(quality.non_numeric_counts(raw).rename("non_numeric"))

{'missing': [], 'unexpected': []}


,dtype,expected,matches,integer_stored_as_float
SeriousDlqin2yrs,int64,integer,True,False
RevolvingUtilizationOfUnsecuredLines,float64,float,True,False
age,int64,integer,True,False
NumberOfTime30-59DaysPastDueNotWorse,int64,integer,True,False
DebtRatio,float64,float,True,False
MonthlyIncome,float64,float,True,False
NumberOfOpenCreditLinesAndLoans,int64,integer,True,False
NumberOfTimes90DaysLate,int64,integer,True,False
NumberRealEstateLoansOrLines,int64,integer,True,False
NumberOfTime60-89DaysPastDueNotWorse,int64,integer,True,False


SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
Name: non_numeric, dtype: int64

### ID, columnas vacías/constantes, target y negativos

In [3]:
print("ID:", quality.id_report(raw))
print("Vacías/constantes:", quality.empty_or_constant_columns(raw))
print("Target:", quality.target_report(raw, cfg.target))
display(quality.negative_counts(raw).rename("negatives"))

ID: {'unique': True, 'nulls': 0, 'non_positive': 0}
Vacías/constantes: {'empty': [], 'constant': []}
Target: {'values': [0, 1], 'nulls': 0}


SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
Name: negatives, dtype: int64

### Valores imposibles o sospechosos por dominio

In [4]:
display(quality.domain_issues(raw))

,check,column,kind,count
0,age < 18,age,impossible,1
1,code 96/98,NumberOfTime30-59DaysPastDueNotWorse,impossible,269
2,code 96/98,NumberOfTimes90DaysLate,impossible,269
3,code 96/98,NumberOfTime60-89DaysPastDueNotWorse,impossible,269
4,== 0,MonthlyIncome,suspicious,1634
5,> 1,RevolvingUtilizationOfUnsecuredLines,suspicious,3321
6,> 1,DebtRatio,suspicious,35137


#### Perfil de los casos de dominio

Por chequeo: filas afectadas, su tasa de mora frente a la base y positivos que se perderían al eliminarlas. Luego, si los códigos 96/98 aparecen en las mismas filas.

In [5]:
display(quality.domain_issue_profile(raw, cfg.target))
print("Filas con 96/98:", quality.past_due_code_overlap(raw))

,check,column,kind,rows,pct_rows,default_rate,base_default_rate,positives
0,age < 18,age,impossible,1,0.000667,0.000000,0.06684,0
1,code 96/98,NumberOfTime30-59DaysPastDueNotWorse,impossible,269,0.179333,0.546468,0.06684,147
2,code 96/98,NumberOfTimes90DaysLate,impossible,269,0.179333,0.546468,0.06684,147
3,code 96/98,NumberOfTime60-89DaysPastDueNotWorse,impossible,269,0.179333,0.546468,0.06684,147
4,== 0,MonthlyIncome,suspicious,1634,1.089333,0.040392,0.06684,66
5,> 1,RevolvingUtilizationOfUnsecuredLines,suspicious,3321,2.214000,0.372478,0.06684,1237
6,> 1,DebtRatio,suspicious,35137,23.424667,0.065202,0.06684,2291


Filas con 96/98: {'any': 269, 'all': 269}


### Duplicados y comparación con la referencia de la spec

In [6]:
exact = quality.count_exact_duplicates(raw)
conflicting = quality.count_conflicting_duplicate_groups(raw, cfg.target)
empty = len(quality.empty_or_constant_columns(raw)["empty"])
negatives = int(quality.negative_counts(raw).sum())
reference = pd.DataFrame(
    {
        "referencia": [609, 37, 0, 0],
        "observado": [exact, conflicting, empty, negatives],
    },
    index=["duplicados exactos", "grupos contradictorios", "columnas vacías", "valores negativos"],
)
reference["coincide"] = reference["referencia"] == reference["observado"]
display(reference)

,referencia,observado,coincide
duplicados exactos,609,609,True
grupos contradictorios,37,37,True
columnas vacías,0,0,True
valores negativos,0,0,True


### Única acción: eliminar duplicados exactos

In [7]:
dedup = quality.drop_exact_duplicates(raw)
removed = len(raw) - len(dedup)
print(f"Filas antes: {len(raw):,} · después: {len(dedup):,} · eliminadas: {removed:,}")

Filas antes: 150,000 · después: 149,391 · eliminadas: 609


## 2. Split estratificado

In [8]:
parts = split_train_valid_test(
    dedup, cfg.target, cfg.split.test_size, cfg.split.valid_size, cfg.seed
)
summary = partition_summary(parts, cfg.target)
display(summary)
spread_pp = (summary["default_rate"].max() - summary["default_rate"].min()) * 100
print(f"Diferencia máxima de tasa de mora entre particiones: {spread_pp:.4f} pp")
overlap = cross_partition_feature_overlap(parts, FEATURES)
print("Combinaciones de variables en más de una partición:", overlap)
save_partitions(parts, cfg.paths.interim);

,rows,default_rate
train,95609,0.067002
valid,23903,0.066979
test,29879,0.067004


Diferencia máxima de tasa de mora entre particiones: 0.0025 pp


Combinaciones de variables en más de una partición:

 19


## 3. Fase 2 — EDA estadístico (solo train interno)

A partir de aquí **ninguna celda usa `parts["valid"]` ni `parts["test"]`**. Ningún resultado modifica los datos.

In [9]:
train = parts["train"]
print(f"Train interno: {len(train):,} filas")

Train interno: 95,609 filas


### Target

In [10]:
display(pd.Series(profile.target_summary(train, cfg.target)))
fig = plots.plot_target_distribution(train, cfg.target)
plots.save_figure(fig, FIG_DIR / "target_distribution.png");

rows            95609.000000
positives        6406.000000
default_rate        0.067002
dtype: float64

### Nulos

In [11]:
missing = profile.missing_summary(train, cfg.target)
display(missing)
plots.save_figure(plots.plot_missing(missing), FIG_DIR / "missing.png");

,missing_count,missing_pct,default_rate_missing,default_rate_present
RevolvingUtilizationOfUnsecuredLines,0,0.000000,NaN,0.067002
age,0,0.000000,NaN,0.067002
NumberOfTime30-59DaysPastDueNotWorse,0,0.000000,NaN,0.067002
DebtRatio,0,0.000000,NaN,0.067002
MonthlyIncome,18736,19.596482,0.054814,0.069973
NumberOfOpenCreditLinesAndLoans,0,0.000000,NaN,0.067002
NumberOfTimes90DaysLate,0,0.000000,NaN,0.067002
NumberRealEstateLoansOrLines,0,0.000000,NaN,0.067002
NumberOfTime60-89DaysPastDueNotWorse,0,0.000000,NaN,0.067002
NumberOfDependents,2451,2.563566,0.044472,0.067595


### Outliers

In [12]:
display(profile.percentile_table(train, FEATURES))
display(profile.outlier_counts(train, FEATURES))

,p1,p5,p25,p50,p75,p95,p99,max
RevolvingUtilizationOfUnsecuredLines,0.0,0.000000,0.030152,0.153928,0.556722,1.0,1.0938,50708.0
age,24.0,29.000000,41.000000,52.000000,63.000000,78.0,87.0000,109.0
NumberOfTime30-59DaysPastDueNotWorse,0.0,0.000000,0.000000,0.000000,0.000000,2.0,3.0000,98.0
DebtRatio,0.0,0.005238,0.176562,0.367434,0.869917,2455.0,4982.6800,329664.0
MonthlyIncome,0.0,1308.000000,3401.000000,5416.000000,8291.000000,14667.8,25000.0000,3008750.0
NumberOfOpenCreditLinesAndLoans,0.0,2.000000,5.000000,8.000000,11.000000,18.0,24.0000,58.0
NumberOfTimes90DaysLate,0.0,0.000000,0.000000,0.000000,0.000000,1.0,2.0000,98.0
NumberRealEstateLoansOrLines,0.0,0.000000,0.000000,1.000000,2.000000,3.0,5.0000,54.0
NumberOfTime60-89DaysPastDueNotWorse,0.0,0.000000,0.000000,0.000000,0.000000,0.6,2.0000,98.0
NumberOfDependents,0.0,0.000000,0.000000,0.000000,1.000000,3.0,4.0000,13.0


,iqr_outliers,above_p99
RevolvingUtilizationOfUnsecuredLines,492,957
age,32,774
NumberOfTime30-59DaysPastDueNotWorse,15239,951
DebtRatio,20051,957
MonthlyIncome,3110,741
NumberOfOpenCreditLinesAndLoans,2497,948
NumberOfTimes90DaysLate,5239,928
NumberRealEstateLoansOrLines,506,506
NumberOfTime60-89DaysPastDueNotWorse,4781,457
NumberOfDependents,8522,651


### `DebtRatio` según el estado del ingreso

Sin ingreso no se puede calcular el ratio. Si en esas filas `DebtRatio` es siempre entero y del orden de miles, la columna guarda un **monto** de deuda y no un ratio.

In [13]:
display(profile.debt_ratio_by_income_status(train, cfg.target).round(3))

,rows,debt_ratio_median,pct_debt_ratio_gt_1,pct_debt_ratio_integer,default_rate
missing,18736,1204.000,95.581,100.000,0.055
zero,1048,958.500,93.607,100.000,0.041
positive,75825,0.291,4.662,2.191,0.070


### Distribuciones por target

In [14]:
log_columns = [*RATIO_COLUMNS, "MonthlyIncome"]
for feature in FEATURES:
    fig = plots.plot_distribution_by_target(train, feature, cfg.target, log=feature in log_columns)
    plots.save_figure(fig, FIG_DIR / f"dist_{feature}.png")
print(f"{len(FEATURES)} figuras guardadas en reports/figures")

10 figuras guardadas en reports/figures


### Correlaciones (Spearman)

In [15]:
corr = profile.spearman_matrix(train, FEATURES)
display(corr.round(2))
plots.save_figure(plots.plot_correlation_heatmap(corr), FIG_DIR / "corr_spearman.png");

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
RevolvingUtilizationOfUnsecuredLines,1.00,-0.28,0.23,0.07,-0.08,-0.09,0.24,-0.03,0.19,0.12
age,-0.28,1.00,-0.10,0.03,0.13,0.16,-0.11,0.05,-0.08,-0.23
NumberOfTime30-59DaysPastDueNotWorse,0.23,-0.10,1.00,0.03,-0.01,0.06,0.25,0.02,0.27,0.07
DebtRatio,0.07,0.03,0.03,1.00,-0.13,0.22,-0.03,0.40,0.00,-0.04
MonthlyIncome,-0.08,0.13,-0.01,-0.13,1.00,0.31,-0.09,0.39,-0.06,0.20
NumberOfOpenCreditLinesAndLoans,-0.09,0.16,0.06,0.22,0.31,1.00,-0.14,0.47,-0.05,0.09
NumberOfTimes90DaysLate,0.24,-0.11,0.25,-0.03,-0.09,-0.14,1.00,-0.10,0.31,0.04
NumberRealEstateLoansOrLines,-0.03,0.05,0.02,0.40,0.39,0.47,-0.10,1.00,-0.04,0.16
NumberOfTime60-89DaysPastDueNotWorse,0.19,-0.08,0.27,0.00,-0.06,-0.05,0.31,-0.04,1.00,0.04
NumberOfDependents,0.12,-0.23,0.07,-0.04,0.20,0.09,0.04,0.16,0.04,1.00


### Tasa de mora por decil

In [16]:
for feature in FEATURES:
    table = profile.default_rate_by_decile(train, feature, cfg.target)
    print(feature)
    display(table)
    fig = plots.plot_default_rate_by_decile(table, feature)
    plots.save_figure(fig, FIG_DIR / f"decile_{feature}.png")

RevolvingUtilizationOfUnsecuredLines


,bin,rows,default_rate
0,"(-0.001, 0.00321]",9561,0.027089
1,"(0.00321, 0.0194]",9561,0.012133
2,"(0.0194, 0.0436]",9561,0.013283
3,"(0.0436, 0.083]",9561,0.019349
4,"(0.083, 0.154]",9561,0.023533
5,"(0.154, 0.271]",9560,0.035669
6,"(0.271, 0.444]",9561,0.050622
7,"(0.444, 0.695]",9561,0.088171
8,"(0.695, 0.977]",9561,0.166196
9,"(0.977, 50708.0]",9561,0.233971


age


,bin,rows,default_rate
0,"(-0.001, 33.0]",10745,0.112983
1,"(33.0, 39.0]",9572,0.099666
2,"(39.0, 44.0]",10064,0.088633
3,"(44.0, 48.0]",9420,0.082590
4,"(48.0, 52.0]",9440,0.076165
5,"(52.0, 56.0]",9095,0.063222
6,"(56.0, 60.0]",8607,0.048333
7,"(60.0, 65.0]",10599,0.039343
8,"(65.0, 72.0]",9023,0.027596
9,"(72.0, 109.0]",9044,0.021230


NumberOfTime30-59DaysPastDueNotWorse


,bin,rows,default_rate
0,"(-0.001, 1.0]",90664,0.053119
1,"(1.0, 98.0]",4945,0.321537


... (10 outputs left out)

NumberOfTime60-89DaysPastDueNotWorse


,bin,rows,default_rate
0,"(-0.001, 98.0]",95609,0.067002


NumberOfDependents


,bin,rows,default_rate
0,"(-0.001, 1.0]",72098,0.062346
1,"(1.0, 2.0]",12538,0.080794
2,"(2.0, 13.0]",8522,0.092584
3,missing,2451,0.044472
